# ARTI501 – Natural Language Processing
# Lab 3 – Part 2: Text Analysis & N-Gram Generation

This notebook covers two tasks:
- **Task #1:** Text analysis and NLP pre-processing on the IMDB 50K Movie Reviews dataset.
- **Task #2:** Bigram generation and frequency analysis on a poem dataset.

ALI ZUHAIR ALSAFFAR 2240005706

## Objective / Learning Outcome

**CLO1:** Fundamentals of NLP — N-Gram models and text pre-processing.

By the end of this notebook we will be able to:
- Tokenize, clean, and remove stopwords from raw text.
- Compute basic corpus statistics (review counts, average length, keyword-based counts).
- Visualize word frequency with a histogram.
- Generate bigrams from text and find the most frequent ones.

## Setup

Run these once (in a terminal, or uncomment and run the cell below):

In [1]:
# Uncomment the lines below the first time you run this notebook
# !pip install nltk
# !pip install matplotlib

In [2]:
import nltk

nltk.download('punkt')
try:
    nltk.download('punkt_tab')
except Exception:
    pass
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aliza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aliza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aliza\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

---
# Task #1 — Text Analysis and NLP (IMDB Movie Reviews)


## 1. Import Libraries

In [3]:
import pandas as pd
import re
import string
from collections import Counter
import matplotlib.pyplot as plt

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

## 2. Load the Dataset

**Dataset:** *IMDB Dataset of 50K Movie Reviews* (Kaggle).

- **Link:** https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

**How to get the file:**
1. Open the Kaggle link above and click **Download**.
2. Place the downloaded CSV file (`IMDB Dataset.csv`) in the same folder as this notebook.
3. Update `DATASET_PATH` below if your filename is different.

In [4]:
DATASET_PATH = "IMDB Dataset.csv"   # update this to match your downloaded file's name

try:
    imdb_df = pd.read_csv(DATASET_PATH)
    print(f"Dataset loaded successfully from '{DATASET_PATH}'. Shape: {imdb_df.shape}")
except FileNotFoundError:
    imdb_df = None
    print(f"Could not find '{DATASET_PATH}'.\n"
          "Please download the dataset from:\n"
          "https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews\n"
          "and place the CSV file in the same folder as this notebook "
          "(update DATASET_PATH above if the filename is different).")

Could not find 'IMDB Dataset.csv'.
Please download the dataset from:
https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
and place the CSV file in the same folder as this notebook (update DATASET_PATH above if the filename is different).


In [5]:
if imdb_df is not None:
    print("Columns:", list(imdb_df.columns))
    display(imdb_df.head())

    # Auto-detect the review-text column (the standard Kaggle file uses 'review')
    candidate_columns = ["review", "Review", "text", "Text"]
    review_column = None
    for col in candidate_columns:
        if col in imdb_df.columns:
            review_column = col
            break

    if review_column is None:
        raise ValueError(f"Could not find a review-text column automatically. "
                          f"Available columns are: {list(imdb_df.columns)}. "
                          f"Please set 'review_column' manually.")
    print(f"Review text column detected: '{review_column}'")

## 3. Pre-process the Text

For each review we:
1. **Tokenize** into words with `word_tokenize()`.
2. **Remove punctuation** by keeping only alphabetic tokens (`token.isalpha()`), which also gets rid of standalone punctuation tokens created by tokenization.
3. **Lowercase** every token.
4. **Remove common English stop words** ("the", "is", "and", ...) using NLTK's stop word list.

We keep a *tokenized-but-not-stopword-filtered* version (for an honest "word count per review") and a *fully pre-processed* version (for the word-frequency histogram), since stop words are still real words of the review even though we exclude them from the frequency analysis.

In [6]:
if imdb_df is not None:
    stop_words = set(stopwords.words("english"))

    def tokenize_words(text):
        # Tokenize, then keep only alphabetic tokens (this removes punctuation tokens and numbers)
        tokens = word_tokenize(str(text))
        return [t.lower() for t in tokens if t.isalpha()]

    def remove_stopwords(words):
        return [w for w in words if w not in stop_words]

    # OPTIONAL: uncomment to work with a random sample if the full 50K reviews are too slow on your machine
    # imdb_df = imdb_df.sample(n=10000, random_state=42).reset_index(drop=True)

    # fillna("") first so any missing review becomes an empty string instead of the literal word 'none'
    # Tokenized (lowercased, punctuation-free) words for every review -- used for word-count stats
    imdb_df["tokens"] = imdb_df[review_column].fillna("").apply(tokenize_words)

    # Fully pre-processed words (stop words removed too) -- used for the frequency histogram
    imdb_df["tokens_no_stopwords"] = imdb_df["tokens"].apply(remove_stopwords)

    display(imdb_df[[review_column, "tokens", "tokens_no_stopwords"]].head())

## 4. Calculate Statistics

In [7]:
if imdb_df is not None:
    # Total number of reviews
    total_reviews = len(imdb_df)

    # Average word count per review (based on tokenized words, before stop-word removal)
    avg_word_count = imdb_df["tokens"].apply(len).mean()

    # Number of positive reviews (containing the word 'good') and negative reviews (containing 'bad')
    # \b...\b matches 'good'/'bad' as a whole word, so 'goodbye' or 'badge' are not falsely counted
    positive_reviews = imdb_df[review_column].str.contains(r"\bgood\b", case=False, na=False).sum()
    negative_reviews = imdb_df[review_column].str.contains(r"\bbad\b", case=False, na=False).sum()

    print(f"Total number of reviews: {total_reviews}")
    print(f"Average word count per review: {avg_word_count:.2f}")
    print(f"Number of reviews containing 'good': {positive_reviews}")
    print(f"Number of reviews containing 'bad': {negative_reviews}")

## 5. Word Frequency Histogram

Using the fully pre-processed tokens (punctuation removed, lowercased, stop words removed), we count how often each word occurs across *all* reviews and plot the most common ones.

In [8]:
if imdb_df is not None:
    # Flatten every review's word list into one big list of words
    all_words = [word for tokens in imdb_df["tokens_no_stopwords"] for word in tokens]

    word_freq = Counter(all_words)
    top_20_words = word_freq.most_common(20)

    words, counts = zip(*top_20_words)

    plt.figure(figsize=(12, 5))
    plt.bar(words, counts, color="darkorange")
    plt.title("Top 20 Most Common Words in IMDB Reviews (stop words removed)")
    plt.xlabel("Word")
    plt.ylabel("Frequency")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 6. Print the Results of the Analysis

In [9]:
if imdb_df is not None:
    print("===== IMDB Reviews — Analysis Summary =====")
    print(f"Total number of reviews: {total_reviews}")
    print(f"Average word count per review: {avg_word_count:.2f}")
    print(f"Reviews containing 'good': {positive_reviews}")
    print(f"Reviews containing 'bad': {negative_reviews}")
    print("\nTop 10 most common words (stop words removed):")
    for rank, (word, count) in enumerate(word_freq.most_common(10), start=1):
        print(f"{rank}. {word} — {count}")

---
# Task #2 — N-gram Generation from Poem Data


## 1. Import Libraries

We reuse `word_tokenize`, `stopwords`, `pandas`, and `Counter` from Task #1, plus NLTK's `bigrams` utility to generate 2-grams from token lists.

In [10]:
from nltk import bigrams

## 2. Load the Dataset

**Dataset:** *Poem Classification (NLP)* (Kaggle).

- **Link:** https://www.kaggle.com/datasets/ishikajohari/poem-classification-nlp-dataset

**How to get the file:**
1. Open the Kaggle link above and click **Download**.
2. Place the training CSV file (commonly named `Poem_classification - train_data.csv`) in the same folder as this notebook.
3. Update `POEM_DATASET_PATH` below if your filename is different.

This dataset has a `Poem` column (the poem text) and a `Genre` column (its category).

In [11]:
POEM_DATASET_PATH = "Poem_classification - train_data.csv"   # update to match your downloaded file's name

try:
    poem_df = pd.read_csv(POEM_DATASET_PATH)
    print(f"Dataset loaded successfully from '{POEM_DATASET_PATH}'. Shape: {poem_df.shape}")
except FileNotFoundError:
    poem_df = None
    print(f"Could not find '{POEM_DATASET_PATH}'.\n"
          "Please download the dataset from:\n"
          "https://www.kaggle.com/datasets/ishikajohari/poem-classification-nlp-dataset\n"
          "and place the CSV file in the same folder as this notebook "
          "(update POEM_DATASET_PATH above if the filename is different).")

Could not find 'Poem_classification - train_data.csv'.
Please download the dataset from:
https://www.kaggle.com/datasets/ishikajohari/poem-classification-nlp-dataset
and place the CSV file in the same folder as this notebook (update POEM_DATASET_PATH above if the filename is different).


In [12]:
if poem_df is not None:
    print("Columns:", list(poem_df.columns))
    display(poem_df.head())

    # Auto-detect the poem-text column
    candidate_columns = ["Poem", "poem", "text", "Text"]
    poem_column = None
    for col in candidate_columns:
        if col in poem_df.columns:
            poem_column = col
            break

    if poem_column is None:
        raise ValueError(f"Could not find a poem-text column automatically. "
                          f"Available columns are: {list(poem_df.columns)}. "
                          f"Please set 'poem_column' manually.")
    print(f"Poem text column detected: '{poem_column}'")

    # Drop any rows with a missing poem
    poem_df = poem_df.dropna(subset=[poem_column]).reset_index(drop=True)

## 3. Pre-process the Text

Same pipeline as Task #1: tokenize, remove punctuation (keep alphabetic tokens only), lowercase, and remove common English stop words.

In [13]:
if poem_df is not None:
    # Reuse the same helper functions defined in Task #1
    poem_df["tokens"] = poem_df[poem_column].apply(tokenize_words)
    poem_df["tokens_no_stopwords"] = poem_df["tokens"].apply(remove_stopwords)

    display(poem_df[[poem_column, "tokens_no_stopwords"]].head())

## 4. Generate Bigrams

For every poem, we generate all 2-grams (bigrams) from its pre-processed word list using NLTK's `bigrams()` function, and store them in a new `bigrams` column.

In [14]:
if poem_df is not None:
    # Generate bigrams for each poem's pre-processed word list
    poem_df["bigrams"] = poem_df["tokens_no_stopwords"].apply(lambda words: list(bigrams(words)))

    display(poem_df[[poem_column, "bigrams"]].head())

## 5. Frequency of Each Unique Bigram

We flatten the `bigrams` column (a list of bigrams per poem) into one list containing every bigram occurrence across the whole dataset, then count how often each unique bigram appears with `Counter`.

In [15]:
if poem_df is not None:
    all_bigrams = [bg for bigram_list in poem_df["bigrams"] for bg in bigram_list]

    bigram_freq = Counter(all_bigrams)

    print(f"Total number of bigrams (occurrences): {len(all_bigrams)}")
    print(f"Number of unique bigrams: {len(bigram_freq)}")

## 6. Top 10 Most Common Bigrams

In [16]:
if poem_df is not None:
    top_10_bigrams = bigram_freq.most_common(10)

    # Display as a clean table -- join each bigram tuple into a readable 'word1 word2' string
    top_10_bigrams_df = pd.DataFrame(
        [(" ".join(bg), count) for bg, count in top_10_bigrams],
        columns=["Bigram", "Frequency"]
    )
    top_10_bigrams_df.index = range(1, len(top_10_bigrams_df) + 1)
    top_10_bigrams_df.index.name = "Rank"

    display(top_10_bigrams_df)

## 7. Results / Conclusion

**Task #1 (IMDB reviews):** We tokenized, cleaned, and stop-word-filtered 50K movie reviews, computed the total review count, average word count per review, and how many reviews mention "good" vs. "bad", then visualized the most frequent words with a histogram.

**Task #2 (Poem bigrams):** We applied the same pre-processing pipeline to a poem dataset, generated bigrams from each poem's cleaned word list, counted how often each unique bigram occurs across the whole corpus, and displayed the 10 most frequent bigrams in a ranked table.

Both tasks show how the same core pre-processing steps (tokenization, punctuation/stop-word removal, lowercasing) feed directly into two different kinds of text analysis: simple corpus statistics, and n-gram frequency analysis.